In [13]:
pip install scikit-learn==1.2.0


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [14]:
import pandas as pd 
import numpy as np
import joblib


In [15]:
model = joblib.load("/Users/sadiqkhawaja/Desktop/PhD/CVD_RiskStrat_WebApp-merge-interface2/model_files/batch_models/TAVIModel.pkl")

In [16]:
exported_forest = []

for tree in model.estimators_:
    exported_forest.append({
        "feature": tree.tree_.feature.copy(),
        "threshold": tree.tree_.threshold.copy(),
        "left": tree.tree_.children_left.copy(),
        "right": tree.tree_.children_right.copy(),
        "value": tree.tree_.value[:,0,:].copy()
    })

In [17]:
forest_export = {
    "n_features_in": model.n_features_in_,
    "classes": model.classes_.tolist(),
    "trees": exported_forest
}

In [18]:
backendValues = {
    "Age":75,
    "Diabetes":0,
    "Ex_smoking":0,
    "Hypertension":0,
    "Creatinine":0,
    "Pre-operative_heart_rhythm":0,
    "Baseline_Hb":0,
    "Poor_mobility":0,
    "FEV1":0,
    "Predicted_VC":0,
    "FEV1_VC":0,
    "Katz_Index":0,
    "TR":0,
}

In [19]:
feature_order = [
    "Age",
    "Diabetes",
    "Ex_smoking",
    "Hypertension",
    "Creatinine",
    "Pre-operative_heart_rhythm",
    "Baseline_Hb",
    "Poor_mobility",
    "FEV1",
    "Predicted_VC",
    "FEV1_VC",
    "Katz_Index",
    "TR"
]

x = np.array([backendValues[f] for f in feature_order])

In [20]:
def predict_tree(tree, x):
    node = 0

    while tree["left"][node] != tree["right"][node]:
        feat = tree["feature"][node]
        thresh = tree["threshold"][node]

        if x[feat] <= thresh:
            node = tree["left"][node]
        else:
            node = tree["right"][node]

    return tree["value"][node]

In [21]:
def predict_forest_sklearn_style(forest_export, x):

    n_classes = len(forest_export["classes"])
    total = np.zeros(n_classes)

    for tree in forest_export["trees"]:

        leaf = predict_tree(tree, x).astype(float)

        # Convert this tree's leaf counts to probabilities
        tree_prob = leaf / leaf.sum()

        total += tree_prob

    return total / len(forest_export["trees"])

In [22]:
sk_prob = model.predict_proba([x])[0]
manual_prob = predict_forest_sklearn_style(forest_export, x)

print("sklearn :", sk_prob)
print("manual  :", manual_prob)
print("max diff:", np.max(np.abs(sk_prob - manual_prob)))
print("sklearn :", sk_prob)
print("manual  :", manual_prob)

print("max diff:", np.max(np.abs(sk_prob - manual_prob)))

sklearn : [0.6038961 0.3961039]
manual  : [0.6038961 0.3961039]
max diff: 0.0
sklearn : [0.6038961 0.3961039]
manual  : [0.6038961 0.3961039]
max diff: 0.0


In [23]:
print("Trees exported:", len(exported_forest))
print("Model trees:", len(model.estimators_))

Trees exported: 154
Model trees: 154
